# import libraries

In [93]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline
import datetime
from sklearn.metrics import mean_squared_error,mean_absolute_error,root_mean_squared_error,r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# load data

In [94]:
data=pd.read_csv("carprice_ml_pro.csv")
df=pd.DataFrame(data)
df

,it,ft,bt,km,transmission,ownerNo,owner,oem,model,modelYear,...,Cargo Volumn,Alloy Wheel Size_2,City,Gear Box_1,Rear Brake Type_1,Front Brake Type_1,Acceleration_1,Top Speed_1,Drive Type_1,Turning Radius_1
0,0,Petrol,Hatchback,"1,20,000",Manual,3,3rd Owner,Maruti,Maruti Celerio,2015,...,235-litres,NaN,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,Petrol,SUV,"32,706",Manual,2,2nd Owner,Ford,Ford Ecosport,2018,...,352-litres,16,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,Petrol,Hatchback,"11,949",Manual,1,1st Owner,Tata,Tata Tiago,2018,...,242-litres,14,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,Petrol,Sedan,"17,794",Manual,1,1st Owner,Hyundai,Hyundai Xcent,2014,...,407-litres,14,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,Diesel,SUV,"60,000",Manual,1,1st Owner,Maruti,Maruti SX4 S Cross,2015,...,353-litres,16,Bangalore,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8364,0,Petrol,Hatchback,"10,000",Manual,1,1st Owner,Maruti,Maruti Celerio,2022,...,313,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8365,0,Petrol,Hatchback,"1,20,000",Manual,1,1st Owner,Maruti,Maruti Alto 800,2014,...,177-litres,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8366,0,Petrol,Sedan,"50,000",Automatic,3,3rd Owner,Mercedes-Benz,Mercedes-Benz C-Class,2011,...,475-litres,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8367,0,Petrol,Hatchback,"40,000",Manual,1,1st Owner,Maruti,Maruti Ritz,2012,...,236-liters,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# create new features

In [95]:
current_year=datetime.datetime.now().year
current_year

2025

In [96]:
df["Car Age"]=current_year-df["modelYear"]
df["Car Age"]

0       10
1        7
2        7
3       11
4       10
        ..
8364     3
8365    11
8366    14
8367    13
8368     8
Name: Car Age, Length: 8369, dtype: int64

In [97]:
df["Length"]=df["Length"].str.replace("mm","").str.replace(",","").str.strip().astype(float)
df["Width"]=df["Width"].str.replace("mm","").str.replace(",","").str.strip().astype(float)
df["Height"]=df["Height"].str.replace("mm","").str.replace(",","").str.replace("-","").str.strip().astype(float)

In [98]:
df["Car Size"]=df["Length"]*df["Width"]*df["Height"]
df["Car Size"].unique()

array([9.50584912e+09, 1.16220061e+10, 9.47043117e+09, 1.00801840e+10,
       1.22424225e+10, 1.31037804e+10, 9.22082775e+09, 1.13491958e+10,
       1.15350345e+10, 1.05266252e+10, 9.49984800e+09, 1.03444449e+10,
       1.54681852e+10, 1.44357199e+10, 1.02813322e+10, 1.16135047e+10,
       1.12510710e+10, 1.31187614e+10, 1.02067448e+10, 1.34649444e+10,
       1.66939261e+10, 9.26398200e+09, 8.64423375e+09, 1.40692669e+10,
       1.36849871e+10, 1.31335956e+10, 9.14685850e+09, 1.28867992e+10,
       1.36495682e+10, 1.53841102e+10, 1.59817800e+10, 1.03995350e+10,
       1.18199025e+10, 1.34506631e+10, 7.69828312e+09, 1.58973872e+10,
       9.40161530e+09, 1.27767150e+10, 1.56674131e+10, 7.52648250e+09,
       2.02795520e+10, 1.57666901e+10, 1.92437940e+10, 8.68370103e+09,
       1.16236463e+10, 9.61062975e+09, 9.50917488e+09, 1.09243593e+10,
       1.23889780e+10, 1.48568921e+10, 1.01934720e+10, 1.32979542e+10,
       1.33655686e+10, 1.25037569e+10, 1.02740400e+10, 1.06721431e+10,
      

In [99]:
selected_feature=['Car Age','Car Size','ft', 'bt', 'km', 'transmission', 'ownerNo','oem',
                  'model', 'modelYear', 'price','Insurance Validity','Mileage', 'Engine',
                  ' Seats','Color','City']


# drop unwanted columns

In [100]:
new_df=df.drop(columns=['it', 'owner',  'centralVariantId', 'variantName',
       'priceActual', 'priceSaving', 'priceFixedText', 'trendingText.imgUrl',
       'trendingText.heading', 'trendingText.desc', 'Registration Year',
       'Fuel Type', 'Kms Driven', 'RTO',
       'Ownership', 'Engine Displacement', 'Transmission',
       'Year of Manufacture', 'Max Power', 'Torque',
       'Seats_1', 'Wheel Size', 'Engine Type', 'Displacement',
       'Max Power_1', 'Max Torque', 'No of Cylinder', 'Values per Cylinder',
       'Value Configuration', 'Fuel Suppy System', 'BoreX Stroke',
       'Compression Ratio', 'Turbo Charger', 'Super Charger',
       'Seating Capacity', 'Steering Type', 'Tyre Type', 'Alloy Wheel Size',
       'No Door Numbers', 'Length', 'Width', 'Height', 'Wheel Base',
       'Front Tread', 'Rear Tread', 'Kerb Weight', 'Gross Weight',
       'Ground Clearance Unladen', 'Seating Capacity_1', 'Steering Type_1',
       'Tyre Type_1', 'Alloy Wheel Size_1', 'No Door Numbers_1', 'Gear Box',
       'Drive Type', 'Seating Capacity_2', 'Steering Type_2', 'Turning Radius',
       'Front Brake Type', 'Rear Brake Type', 'Top Speed', 'Acceleration',
       'Tyre Type_2', 'No Door Numbers_2', 'Cargo Volumn',
       'Alloy Wheel Size_2', 'Gear Box_1', 'Rear Brake Type_1',
       'Front Brake Type_1', 'Acceleration_1', 'Top Speed_1', 'Drive Type_1',
       'Turning Radius_1'],inplace=True)
new_df

# handle numerical columns

In [101]:
df["km"]=df["km"].str.replace(",","").str.strip().astype(float)
df["Insurance Validity"]=df["Insurance Validity"].str.replace("2","Second Party").str.replace("1","First Party").str.strip()
df["Seats"]=df["Seats"].str.replace("Seats","").str.strip().astype(float)
df["Mileage"]=df["Mileage"].str.replace("kmpl","").str.replace("km/kg",'').str.strip().astype(float)
df["Engine"]=df["Engine"].str.replace("CC","").str.strip().astype(float)
df["price"]=df["price"].str.replace("Lakh","").str.replace("Crore","").str.replace(",","").str.replace("₹","").str.strip().astype(float)

# drop duplicate values

In [102]:
df.drop_duplicates(inplace=True)

In [103]:
df.dropna(subset=["Mileage"],inplace=True)
df.shape

(7975, 17)

In [104]:
df.dropna(subset=["Car Size"],inplace=True)
df.shape

(7915, 17)

# handle missing values

In [105]:
df["bt"].fillna(df["bt"].mode()[0],inplace=True)
df["Insurance Validity"].fillna(df["Insurance Validity"].mode()[0],inplace=True)
df["Seats"].fillna(df["Seats"].median(),inplace=True)
df["Engine"].fillna(df["Engine"].mode()[0],inplace=True)
df["Color"].fillna(df["Color"].mode()[0],inplace=True)

# handle outliers

In [106]:
q1=df["price"].quantile(0.25)
q3=df["price"].quantile(0.75)
iqr=q3-q1
lf=q1-1.5*iqr
uf=q3+1.5*iqr
dff=df[(df["price"]>lf)&(df['price']<uf)]
dff

,ft,bt,km,transmission,ownerNo,oem,model,modelYear,price,Insurance Validity,Seats,Mileage,Engine,Color,City,Car Age,Car Size
0,Petrol,Hatchback,120000.0,Manual,3,Maruti,Maruti Celerio,2015,4.00,Third Party insurance,5.0,23.10,998.0,White,Bangalore,10,9.505849e+09
1,Petrol,SUV,32706.0,Manual,2,Ford,Ford Ecosport,2018,8.11,Comprehensive,5.0,17.00,1497.0,White,Bangalore,7,1.162201e+10
2,Petrol,Hatchback,11949.0,Manual,1,Tata,Tata Tiago,2018,5.85,Comprehensive,5.0,23.84,1199.0,Red,Bangalore,7,9.470431e+09
3,Petrol,Sedan,17794.0,Manual,1,Hyundai,Hyundai Xcent,2014,4.62,Comprehensive,5.0,19.10,1197.0,Others,Bangalore,11,1.008018e+10
4,Diesel,SUV,60000.0,Manual,1,Maruti,Maruti SX4 S Cross,2015,7.90,Third Party insurance,5.0,23.65,1248.0,Gray,Bangalore,10,1.224242e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8364,Petrol,Hatchback,10000.0,Manual,1,Maruti,Maruti Celerio,2022,5.10,Third Party insurance,5.0,25.24,998.0,Others,Kolkata,3,9.509175e+09
8365,Petrol,Hatchback,120000.0,Manual,1,Maruti,Maruti Alto 800,2014,1.80,Third Party insurance,5.0,22.74,796.0,Others,Kolkata,11,7.461361e+09
8366,Petrol,Sedan,50000.0,Automatic,3,Mercedes-Benz,Mercedes-Benz C-Class,2011,5.50,Third Party insurance,5.0,11.74,1796.0,Others,Kolkata,14,1.175842e+10
8367,Petrol,Hatchback,40000.0,Manual,1,Maruti,Maruti Ritz,2012,1.40,Third Party insurance,5.0,18.50,1197.0,Others,Kolkata,13,1.027404e+10


# remane columns

In [107]:
dff=dff.rename(columns={"ft":"Fuel Type","bt":"Body Type","transmission":"Transmission","ownerNo":"Owner No","model":"Model",
                        "modelYear":"Model Year","price":"Price","km":"Kilometer","oem":"Brand"})
dff

,Fuel Type,Body Type,Kilometer,Transmission,Owner No,Brand,Model,Model Year,Price,Insurance Validity,Seats,Mileage,Engine,Color,City,Car Age,Car Size
0,Petrol,Hatchback,120000.0,Manual,3,Maruti,Maruti Celerio,2015,4.00,Third Party insurance,5.0,23.10,998.0,White,Bangalore,10,9.505849e+09
1,Petrol,SUV,32706.0,Manual,2,Ford,Ford Ecosport,2018,8.11,Comprehensive,5.0,17.00,1497.0,White,Bangalore,7,1.162201e+10
2,Petrol,Hatchback,11949.0,Manual,1,Tata,Tata Tiago,2018,5.85,Comprehensive,5.0,23.84,1199.0,Red,Bangalore,7,9.470431e+09
3,Petrol,Sedan,17794.0,Manual,1,Hyundai,Hyundai Xcent,2014,4.62,Comprehensive,5.0,19.10,1197.0,Others,Bangalore,11,1.008018e+10
4,Diesel,SUV,60000.0,Manual,1,Maruti,Maruti SX4 S Cross,2015,7.90,Third Party insurance,5.0,23.65,1248.0,Gray,Bangalore,10,1.224242e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8364,Petrol,Hatchback,10000.0,Manual,1,Maruti,Maruti Celerio,2022,5.10,Third Party insurance,5.0,25.24,998.0,Others,Kolkata,3,9.509175e+09
8365,Petrol,Hatchback,120000.0,Manual,1,Maruti,Maruti Alto 800,2014,1.80,Third Party insurance,5.0,22.74,796.0,Others,Kolkata,11,7.461361e+09
8366,Petrol,Sedan,50000.0,Automatic,3,Mercedes-Benz,Mercedes-Benz C-Class,2011,5.50,Third Party insurance,5.0,11.74,1796.0,Others,Kolkata,14,1.175842e+10
8367,Petrol,Hatchback,40000.0,Manual,1,Maruti,Maruti Ritz,2012,1.40,Third Party insurance,5.0,18.50,1197.0,Others,Kolkata,13,1.027404e+10


# split target and features

In [108]:
x=dff.drop("Price",axis=1)
y=dff["Price"]

# split train and test data

In [109]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# create pipeline

In [110]:
numerical_features=x.select_dtypes(include=np.number).columns.tolist()
categorical_features=x.select_dtypes(exclude=np.number).columns.tolist()

In [111]:
numerical_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])

In [112]:
categorical_transformer=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1))
])

In [113]:
preprocessor=ColumnTransformer(
    transformers=[
        ('num',numerical_transformer,numerical_features),
        ('cat',categorical_transformer,categorical_features)
    ],
    remainder='passthrough'
)

In [114]:
model_pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ('regressor',RandomForestRegressor(n_estimators=100,random_state=42))
])

# fit pipeline

In [115]:
print("training the model.....")
model_pipeline.fit(x_train,y_train)
print("model training complete.")

training the model.....
model training complete.


# model prediction

In [116]:
print("making prediction...")
y_pred=model_pipeline.predict(x_test)
print("prediction completed..")

making prediction...
prediction completed..


# evaluate the model

In [117]:
r2_score=r2_score(y_test,y_pred)
print("r2_score :",r2_score)

r2_score : 0.8858771458250427


In [118]:
x_train_processed=preprocessor.fit_transform(x_train)
print("\n sample of preprocessed training data :",x_train_processed[:5])


 sample of preprocessed training data : [[ 6.51418620e-02 -5.75533924e-01  7.28723635e-01 -2.78319064e-01
   4.73012551e-01 -9.00283176e-01 -7.28723635e-01 -3.02287693e-02
   4.00000000e+00  2.00000000e+00  0.00000000e+00  1.90000000e+01
   1.19000000e+02  4.00000000e+00  4.10000000e+01  3.00000000e+00]
 [ 8.41388469e-01 -5.75533924e-01  1.75058711e-01 -2.78319064e-01
  -2.20004694e-01 -8.97626806e-01 -1.75058711e-01 -2.93845167e-02
   4.00000000e+00  6.00000000e+00  1.00000000e+00  6.00000000e+00
   3.90000000e+01  0.00000000e+00  4.10000000e+01  2.00000000e+00]
 [ 6.53674562e-01 -5.75533924e-01  1.00555610e+00  3.03097681e+00
  -1.04081809e-01  3.32272477e-01 -1.00555610e+00 -2.91092377e-02
   4.00000000e+00  3.00000000e+00  1.00000000e+00  1.90000000e+01
   1.22000000e+02  0.00000000e+00  8.40000000e+01  4.00000000e+00]
 [ 1.51347890e-01  9.98318185e-01 -3.78606213e-01 -2.78319064e-01
   1.90765528e-01 -9.00283176e-01  3.78606213e-01 -3.03476416e-02
   4.00000000e+00  2.00000000e+0

# save as pickle file

In [119]:
import pickle

In [120]:
filename='car_price_prediction.pkl'

In [121]:
with open(filename,'wb') as file:
    pickle.dump(model_pipeline,file)
print(f"model saved as {filename}")

model saved as car_price_prediction.pkl


# load model from the pickle file

In [122]:
with open('car_price_prediction.pkl','rb') as file:
    loaded_model=pickle.load(file)